# Data Querying — EDA over the parquet dataset

Driving example (from `docs/plans/DataPipeline.md`):

> Report a player's performance in matches when a certain other player also
> played, vs their overall average.

**Question:** How did Haaland perform in 2025-26 Premier League matches when
Foden also played, vs when Foden didn't?

This notebook reads only `data/processed/*_2025-2026.parquet` (build with
`python scripts/ingest.py --config config/data.yaml`). Everything is plain
Polars — no query API, per `docs/style/Data.md`.

In [1]:
from pathlib import Path

import polars as pl

import yaml

# locate the project root regardless of the kernel's cwd
root = Path.cwd()
while not (root / "config" / "data.yaml").exists() and root != root.parent:
    root = root.parent

cfg = yaml.safe_load((root / "config" / "data.yaml").read_text())
processed = root / cfg["processed_dir"]
season = cfg["seasons"]["default"]

def t(name: str) -> pl.DataFrame:
    return pl.read_parquet(processed / f"{name}_{season}.parquet")

players, teams = t("players"), t("teams")
gw_stats, match_stats, matches = t("gw_stats"), t("match_stats"), t("matches")

print(f"season={season}  players={players.height}  gw_stats={gw_stats.height}  "
      f"match_stats={match_stats.height}  matches={matches.height}")

season=2025-2026  players=841  gw_stats=29978  match_stats=15340  matches=525


## 1. Resolve players by name

Name resolution is just a filter over `players.parquet` — the data is the
index (no `find_player` API).

In [2]:
players.filter(pl.col("web_name").str.to_lowercase().str.contains("haaland")).join(
    teams.select(pl.col("code").alias("team_code"), pl.col("name").alias("team")), on="team_code"
).select("player_code", "web_name", "team", "position")

player_code,web_name,team,position
i64,str,str,str
223094,"""Haaland""","""Man City""","""FWD"""


## 2. Per-GW points for Haaland (his season, five rows)

In [3]:
HAALAND = 223094  # stable player_code across seasons

gw_stats.filter(
    pl.col("player_id").is_in(
        players.filter(pl.col("player_code") == HAALAND).select("player_id").to_series().implode()
    )
).sort("gw").head(5)

player_id,gw,web_name,second_name,status,total_points,minutes,goals_scored,assists,bonus,bps,saves,starts,now_cost,form,ep_next,ep_this,selected_by_percent,season
i64,i64,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,str
430,1,"""Haaland""","""Haaland""","""a""",13,72,2,0,3,49,0,1,15.0,5.0,5.5,5.5,73.0,"""2025-2026"""
430,2,"""Haaland""","""Haaland""","""a""",2,90,0,0,0,6,0,1,14.1,7.5,8.0,8.0,30.7,"""2025-2026"""
430,3,"""Haaland""","""Haaland""","""a""",9,90,1,0,3,29,0,1,14.1,8.0,8.5,8.5,31.9,"""2025-2026"""
430,4,"""Haaland""","""Haaland""","""a""",13,86,2,0,3,63,0,1,14.1,9.2,9.2,9.7,34.7,"""2025-2026"""
430,5,"""Haaland""","""Haaland""","""a""",9,75,1,0,3,32,0,1,14.2,8.2,9.2,8.2,40.8,"""2025-2026"""


## 3. Haaland's played Premier League matches with/without Foden

Semantics: "played" = `minutes_played > 0`; FPL-points stats are Premier League
only — a cup match carries a folder-`gw` and would wrongly join a `gw_stats`
row, so we filter `matches.tournament == 'prem'` first.

In [4]:
prem = matches.filter(pl.col("tournament") == "prem")["match_id"].to_list()
haaland = players.filter(pl.col("player_code") == HAALAND)["player_id"].item()
foden = players.filter(pl.col("web_name") == "Foden")["player_id"].item()

played = match_stats.filter(
    (pl.col("minutes_played") > 0) & pl.col("match_id").is_in(prem)
)
foden_played = played.filter(pl.col("player_id") == foden)["match_id"].to_list()

haaland_matches = played.filter(pl.col("player_id") == haaland).select("match_id", "gw")
perf = haaland_matches.join(
    gw_stats.filter(pl.col("player_id") == haaland).select("gw", "total_points"),
    on="gw",
).with_columns(pl.col("match_id").is_in(foden_played).alias("with_foden"))

def summarize(df: pl.DataFrame) -> None:
    n = df.height
    mean = df["total_points"].mean()
    print(f"   n={n}  mean={mean:.3f}" if n else "   n=0")

print("Haaland FPL points (prem 2025-26):")
print("  with Foden:  ", end="")
summarize(perf.filter(pl.col("with_foden")))
print("  without Foden:", end="")
summarize(perf.filter(~pl.col("with_foden")))
print("  overall:     ", end="")
summarize(perf)

Haaland FPL points (prem 2025-26):
  with Foden:     n=31  mean=6.645
  without Foden:   n=4  mean=11.500
  overall:        n=35  mean=7.200


## Reading the result

- Means come with match counts — a 4-match `without` bucket is a small sample;
  don't over-conclude from it.
- The same join works for any `gw_stats` stat; for match-level stats (xG etc.)
  drop the `gw_stats` join and read `match_stats` directly.
- `player_code` (223094) is stable across seasons; `player_id` is season-local.